# W0a: Course Setup and System Check

This onboarding notebook verifies the complete course toolchain before the first class meeting. It activates the shared Julia environment, loads an external package, includes local course code, generates data, renders a terminal plot, and runs a test cell that checks every one of those layers.

> __Learning Objectives:__
>
> By the end of this system check, you should be able to:
> * __Run a notebook end to end:__ Execute a Julia notebook from top to bottom in the shared course environment. A run that completes without error is the evidence that your installation is usable for course work.
> * __Confirm packages and local code load:__ Verify that both installed packages and the source file in `src/` are visible after the local `Include.jl` runs. These are two separate layers, and each one fails in its own way.
> * __Establish a diagnostic baseline:__ Use this notebook later to distinguish a broken installation from a problem in newly written code. A notebook that passed once and fails now points at the environment rather than at the code you just wrote.

Complete the installation steps in the Week 0 README before running this notebook.
___


## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the pinned course environment, loads the packages used by this system check, and includes a small local source file.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The second line guards against the case where the notebook front end started in a different directory and resolved a different setup file. If this first cell fails, stop and use the troubleshooting table in the Week 0 README rather than continuing with a partially configured environment.

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # activate the pinned environment and load the W0a dependencies
isdefined(Main, :CHEME5800_W0A_ROOT) ||
    error("Setup resolved the wrong Include.jl. Open this notebook from inside its own W0a folder, then restart the kernel.");

The setup cell completed, so the environment, the packages, and the local source are loaded. What follows is a smoke test: a short run that exercises every layer of the toolchain in order, so that a failure anywhere is visible before the first class meeting. The next two tasks exercise an installed package and a local source file, and the test cell at the end checks the result of each.

___


## Task 1: Sampling and plotting
In this task, we test the installation by drawing samples from [a normal distribution](https://en.wikipedia.org/wiki/Normal_distribution) and then visualizing them. This exercises an installed package together with Julia's own random number generator.

> __What is the `let` block?__ The [`let` block](https://docs.julialang.org/en/v1/base/base/#let) creates a new hard scope and optionally introduces new local bindings. Variables introduced inside a `let` block are local to that block and don't affect variables of the same name in the outer scope. In our case, the `let` block allows us to create local variables (`number_of_samples` and the local `samples`) that are only visible within the block, while the final value of `samples` is returned and assigned to the global variable `samples`. This is a common Julia pattern for organizing code and avoiding namespace pollution.

We use [the built-in `randn(...)` function](https://docs.julialang.org/en/v1.11/stdlib/Random/#Base.randn) to draw from a standard normal distribution (mean 0, variance 1). The cell below stores the draws in `samples::Vector{Float64}` for use by the plot and the tests.

In [ ]:
samples = let

    # initialize -
    number_of_samples = 10_000; # set the number of samples we want to generate
    samples = randn(number_of_samples);

    samples; # return
end;

Now, let's plot the `samples::Vector{Float64}` array using [the `histogram(...)` function exported by the `UnicodePlots.jl` package](https://juliaplots.org/UnicodePlots.jl/dev/api/#UnicodePlots.histogram-Tuple%7BAbstractArray%7D).

In [ ]:
let

    # initialize -
    data = samples; # random array
    number_of_bins = 20; # how many bins?
    vertical = false; # vertical or horizontal?
    closed = :left; # which side of the interval is closed?

    # make the histogram -
    histogram(data, nbins=number_of_bins, vertical = vertical, closed = closed);
end

A histogram should appear above, with the counts peaking near zero and falling off in both directions. Confirm that you can see it, then record that in the flag below: the test cell checks `do_you_see_the_histogram::Bool`, which is the one value in this notebook that you set yourself.


In [ ]:
do_you_see_the_histogram = false; # TODO: set this to true once you see the histogram above

___

## Task 2: Loading local source
In this task, we check that methods that we wrote (that were included when we called the `Include.jl` file) are now visible. [The `src/HelloWorld.jl` file](src/HelloWorld.jl) defines the `printgreeting()` function, which __returns__ the string `"Hello World!"`.

> __Return, not print:__ Despite the name, `printgreeting()` has no side effect; it prints nothing. The notebook shows the string because the cell _displays the value of its last expression_. That distinction matters as soon as you start composing functions: a printed value is gone, a returned value can be used.

We'll save the returned value in the `message_that_we_get::String` variable:

In [ ]:
message_that_we_get = printgreeting() # returns "Hello World!"; the notebook displays it

### System-check tests

Run the test cell after you can see the histogram and have changed `do_you_see_the_histogram` to `true`. A completely green result is your setup baseline.


In [ ]:
let
    @testset verbose = true "CHEME 4/5800 Week 0 System Check" begin
        @testset "Environment and local source" begin
            @test isdefined(Main, :CHEME5800_W0A_ROOT)
            @test isfile(joinpath(CHEME5800_W0A_ROOT, "src", "HelloWorld.jl"))
            @test @isdefined printgreeting
            @test message_that_we_get == "Hello World!"
            @test printgreeting() == "Hello World!"
        end

        @testset "Package, data, and plot workflow" begin
            @test @isdefined samples
            @test samples isa Vector{Float64}
            @test length(samples) == 10_000
            @test all(isfinite, samples)
            @test abs(Statistics.mean(samples)) < 0.1
            @test abs(Statistics.std(samples) - 1.0) < 0.1
            @test @isdefined histogram
            @test histogram(samples, nbins = 20) isa UnicodePlots.Plot
            @test do_you_see_the_histogram === true
        end
    end
end;


___


## Summary

A smoke test settles one question before any course work starts: the toolchain on this machine runs the course material end to end.

> __Key Takeaways:__
>
> * **One include connects everything:** The local `Include.jl` delegates to the course root file, which activates the pinned environment, and then loads this meeting's packages and local source. A single line in the first cell is what makes every later cell resolvable.
> * **A smoke test exercises every layer:** Packages, local source, computation, plotting, and tests have to work together, not merely one at a time. A check that skips a layer cannot tell you that layer is broken.
> * **A clean run is a baseline:** Re-run this notebook later to separate environment failures from errors in new work. If it passed before and fails now, suspect the installation rather than the code you just wrote.

Week 1 begins with `L1a`, which assumes the environment this notebook just verified.
___
